In [ ]:
!pip install -q sentence-transformers faiss-cpu rank_bm25 pyvi transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.8 MB/s eta 0:00:00


In [ ]:
# CELL 2: CẤU HÌNH & KHỞI TẠO TỪ HUGGING FACE
import os
from transformers import AutoTokenizer

DRIVE_WORKSPACE = "/content/drive/MyDrive/vimedaq-project"
os.makedirs(os.path.join(DRIVE_WORKSPACE, "data/raw"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_WORKSPACE, "data/processed"), exist_ok=True)

RAW_DATA_PATH = os.path.join(DRIVE_WORKSPACE, "data/raw/vimedaq_full.json")
CORPUS_OUTPUT = os.path.join(DRIVE_WORKSPACE, "data/processed/medical_corpus_V5.json")
BM25_OUTPUT   = os.path.join(DRIVE_WORKSPACE, "data/processed/bm25_index_V5.pkl")
FAISS_OUTPUT  = os.path.join(DRIVE_WORKSPACE, "data/processed/faiss_index_V5.bin")

print("⏳ Đang nạp Tokenizer từ Hugging Face...")
QA_MODEL_NAME = "ntthanh0307/vit5-vimedaq-medical-qa"

MY_HF_TOKEN = "MyToken"

tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME, token=MY_HF_TOKEN)

if not os.path.exists(RAW_DATA_PATH):
    print(f" CẢNH BÁO: Không tìm thấy file gốc tại {RAW_DATA_PATH}.")
else:
    print(f" Đã tìm thấy file dữ liệu gốc. Mọi thứ đã sẵn sàng để xử lý!")

⏳ Đang nạp Tokenizer từ Hugging Face...


config.json:   0%|          | 0.00/812 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.40k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/25.2k [00:00<?, ?B/s]

✅ Đã tìm thấy file dữ liệu gốc. Mọi thứ đã sẵn sàng để xử lý!


In [ ]:
# 1. HÀM CHUNKING BẰNG REGEX (CHỈ DÙNG TOKENIZER ĐỂ ĐẾM)
def safe_sentence_chunking(text, tokenizer, max_tokens=180, overlap_sentences=1):
    if not text or len(text.strip()) < 10:
        return []

    # Tách văn bản thành các câu giữ nguyên dấu câu
    raw_sentences = re.split(r'(?<=[.!?\n])\s+', text.strip())
    sentences = [s.strip() for s in raw_sentences if s.strip()]

    chunks = []
    current_chunk = []
    current_token_count = 0

    for sentence in sentences:
        # ĐẾM token, không thay đổi nội dung câu
        sentence_tokens = len(tokenizer.encode(sentence, add_special_tokens=False))

        # Nếu chỉ 1 câu mà đã vượt ngưỡng (hiếm gặp, nhưng phải bắt lỗi)
        if sentence_tokens > max_tokens:
            if current_chunk:
                chunks.append(" ".join(current_chunk))
                current_chunk = []
                current_token_count = 0
            # Câu quá dài: đành cắt theo số lượng chữ (word-level) chứ không cắt bằng tokenizer
            words = sentence.split()
            temp_sentence = []
            temp_count = 0
            for w in words:
                w_tk = len(tokenizer.encode(w + " ", add_special_tokens=False))
                if temp_count + w_tk > max_tokens and temp_sentence:
                    chunks.append(" ".join(temp_sentence))
                    temp_sentence = []
                    temp_count = 0
                temp_sentence.append(w)
                temp_count += w_tk
            if temp_sentence:
                current_chunk = temp_sentence
                current_token_count = temp_count
            continue

        if current_token_count + sentence_tokens > max_tokens and current_chunk:
            chunks.append(" ".join(current_chunk))

            # Xử lý Overlap
            overlap_start = max(0, len(current_chunk) - overlap_sentences)
            current_chunk = current_chunk[overlap_start:]
            # Đếm lại số token của phần được giữ lại
            current_token_count = len(tokenizer.encode(" ".join(current_chunk), add_special_tokens=False))

        current_chunk.append(sentence)
        current_token_count += sentence_tokens

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

# 2. XỬ LÝ DỮ LIỆU
print("⏳ Đang đọc dữ liệu gốc...")
with open(RAW_DATA_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

unique_contexts = {}
processed_corpus = []
chunk_count = 0

print("⏳ Đang tiến hành Chunking an toàn...")
for item in tqdm(raw_data):
    context = item.get("content") or item.get("text") or item.get("context", "").strip()
    if not context or len(context) < 10:
        continue

    if context not in unique_contexts:
        title = item.get("title", "Không có tiêu đề").strip()
        keyword = item.get("keyword", "Y Tế").strip()
        topic = item.get("topic", "Chung")
        url = item.get("article_url") or item.get("url", "")

        # Cắt câu AN TOÀN
        context_chunks = safe_sentence_chunking(
            context,
            tokenizer=tokenizer,
            max_tokens=180,
            overlap_sentences=1
        )

        for chunk_idx, chunk_text in enumerate(context_chunks):
            # Không cho phép sinh ra chunk nào chỉ toàn ký tự rác
            if len(chunk_text.strip()) < 5:
                continue

            enriched_text = f"[{keyword}] {title}. {chunk_text}"
            final_token_len = len(tokenizer.encode(enriched_text, add_special_tokens=False))

            processed_corpus.append({
                "id": f"doc_{chunk_count}",
                "text": enriched_text,
                "original_context": chunk_text,
                "full_context_source": context,
                "metadata": {
                    "title": title,
                    "keyword": keyword,
                    "topic": topic,
                    "url": url,
                    "chunk_idx": chunk_idx,
                    "token_length": final_token_len
                }
            })
            chunk_count += 1

        unique_contexts[context] = True

# Thống kê an toàn
oversized_chunks = sum(1 for doc in processed_corpus if doc["metadata"]["token_length"] > 256)
unk_chunks = sum(1 for doc in processed_corpus if "<unk>" in doc["text"])

print("\n" + "="*50)
print(" BÁO CÁO KẾT QUẢ TIỀN XỬ LÝ (V5.1):")
print(f"• Số lượng Chunks đẻ ra         : {len(processed_corpus)}")
print(f"• Chunks lỗi dài (>256 tk)      : {oversized_chunks} (Lý tưởng: 0)")
print(f"• Chunks lỗi chứa <unk>         : {unk_chunks} (Lý tưởng: 0)")
print("="*50)

# 
# 3. LƯU & TẠO INDEX
print(f"\n Đang lưu Corpus V5 vào Drive...")
with open(CORPUS_OUTPUT, 'w', encoding='utf-8') as f:
    json.dump(processed_corpus, f, ensure_ascii=False, indent=2)

print("🔍 Đang Tokenize BM25 Index...")
tokenized_corpus = []
for doc in tqdm(processed_corpus, desc="PyVi Tokenizing"):
    tokenized_text = ViTokenizer.tokenize(doc["text"].lower()).split()
    tokenized_corpus.append(tokenized_text)

bm25 = BM25Okapi(tokenized_corpus)
with open(BM25_OUTPUT, 'wb') as f:
    pickle.dump(bm25, f)

print(f"\nĐang tải mô hình Bi-Encoder...")
model = SentenceTransformer("keepitreal/vietnamese-sbert")

print("Đang mã hóa vector (Embedding)...")
texts_to_encode = [doc["text"] for doc in processed_corpus]
embeddings = model.encode(texts_to_encode, show_progress_bar=True, convert_to_numpy=True)

faiss.normalize_L2(embeddings)
faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)
faiss.write_index(faiss_index, FAISS_OUTPUT)

print("\n HOÀN TẤT!")

⏳ Đang đọc dữ liệu gốc...
⏳ Đang tiến hành Chunking an toàn...


  0%|          | 0/44313 [00:00<?, ?it/s]


📊 BÁO CÁO KẾT QUẢ TIỀN XỬ LÝ (V5.1):
• Số lượng Chunks đẻ ra         : 35465
• Chunks lỗi dài (>256 tk)      : 123 (Lý tưởng: 0)
• Chunks lỗi chứa <unk>         : 0 (Lý tưởng: 0)

⏳ Đang lưu Corpus V5 vào Drive...
🔍 Đang Tokenize BM25 Index...


PyVi Tokenizing:   0%|          | 0/35465 [00:00<?, ?it/s]


🧠 Đang tải mô hình Bi-Encoder...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⚡ Đang mã hóa vector (Embedding)...


Batches:   0%|          | 0/1109 [00:00<?, ?it/s]


🎉 HOÀN TẤT! Dữ liệu đã sạch bóng <unk>!
